In [1]:
import xarray as xr
import pandas as pd
import numpy as np
import geopandas as gpd
import json
import matplotlib.pyplot as plt
 
from xcube.core.store import new_data_store
from xcube.core.chunk import chunk_dataset
from xcube.core.gridmapping import GridMapping
from xcube.core.geom import mask_dataset_by_geometry
from xcube_resampling.spatial import resample_in_space
from xcube_resampling.gridmapping import GridMapping

In [2]:
INPUT_DIR = "input_irrigation"

In [3]:
irr_store = new_data_store("file", root=INPUT_DIR)

In [4]:
bbox = [-5, 40, 3, 44] # Ebro Basin
# time_range = ("2020-01-01", "2021-12-31")
time_range = ("2020-01-01", "2020-01-31")

In [5]:
store_lccs = new_data_store("cds", normalize_names=True)

In [6]:
lc = store_lccs.open_data(
        "satellite-land-cover",
        bbox=bbox,
        time_range=time_range,
    )

xcube-cds version 1.2.0
2026-02-16 14:12:21,182 INFO [2025-07-04T00:00:00] Due to a transition between project phases, there are changes to the timeline of this dataset updates, which are usually on an annual basis with a one year delay: 2023 and 2024 data updates are now expected during 2026. Please watch the [forum](https://forum.ecmwf.int/c/announcements/5) for future announcements.
2026-02-16 14:12:21,183 INFO Request ID is 32a605ef-5e17-432b-8903-28fb91972a35
2026-02-16 14:12:21,245 INFO status has been updated to accepted
2026-02-16 14:12:42,451 INFO status has been updated to successful


6583cf58821a0ab48164575888cddc0e.zip:   0%|          | 0.00/4.71M [00:00<?, ?B/s]

In [7]:
lc = lc.sel(time="2020-01-01")
lc = lc[["crs", "lccs_class"]]
lc

<xarray.Dataset> Size: 4MB
Dimensions:     (lat: 1440, lon: 2880)
Coordinates:
  * lat         (lat) float64 12kB 44.0 44.0 43.99 43.99 ... 40.01 40.0 40.0
  * lon         (lon) float64 23kB -4.999 -4.996 -4.993 ... 2.993 2.996 2.999
    time        datetime64[ns] 8B 2020-01-01
Data variables:
    crs         int32 4B ...
    lccs_class  (lat, lon) uint8 4MB dask.array<chunksize=(1440, 2880), meta=np.ndarray>
Attributes: (12/38)
    title:                      Land Cover Map of 2020
    summary:                    This dataset characterizes the land cover of ...
    type:                       C3S-LC-L4-LCCS-Map-300m-P1Y
    references:                 https://cds.climate.copernicus.eu/
    institution:                UCLouvain
    contact:                    copernicus-support@ecmwf.int
    ...                         ...
    geospatial_lon_resolution:  0.002778
    id:                         C3S-LC-L4-LCCS-Map-300m-P1Y-2020-v2.1.1
    project:                    EC C3S Land Cover
    source:                     Sentinel-3 OLCI
    geospatial_lon_min:         -180.0
    geospatial_lon_max:         180.0

In [8]:
%%time
irr_store.write_data(lc, "landcover2020global.zarr")

CPU times: user 45 ms, sys: 23.9 ms, total: 68.9 ms
Wall time: 78.4 ms


'landcover2020global.zarr'

In [9]:
irr_store.list_data_ids()

['landcover2020global.zarr']

In [10]:
irr_store.open_data("landcover2020global.zarr")

<xarray.Dataset> Size: 4MB
Dimensions:     (lat: 1440, lon: 2880)
Coordinates:
  * lat         (lat) float64 12kB 44.0 44.0 43.99 43.99 ... 40.01 40.0 40.0
  * lon         (lon) float64 23kB -4.999 -4.996 -4.993 ... 2.993 2.996 2.999
    time        datetime64[ns] 8B ...
Data variables:
    crs         int32 4B ...
    lccs_class  (lat, lon) uint8 4MB dask.array<chunksize=(1440, 2880), meta=np.ndarray>
Attributes: (12/38)
    Conventions:                CF-1.6
    TileSize:                   2025:2025
    cdm_data_type:              grid
    comment:                    
    contact:                    copernicus-support@ecmwf.int
    creation_date:              20210518T101540Z
    ...                         ...
    time_coverage_end:          20201231
    time_coverage_resolution:   P1Y
    time_coverage_start:        20200101
    title:                      Land Cover Map of 2020
    tracking_id:                5f610500-394f-4f20-b16b-23360e11e968
    type:                       C3S-LC-L4-LCCS-Map-300m-P1Y